## Vera - Seg-Centerline MT Correction (LB 0.45134)

A clean, reproducible entry for the UMUD muscle-architecture challenge. Fork, Run All, and submit `submission.csv` - it scores **0.45134** on the public leaderboard.

### What it predicts
Per ultrasound image: pennation angle (PA), fascicle length (FL), and muscle thickness (MT).

### The idea
PA and FL come from the baseline entry. **Muscle thickness is refined by segmentation.** A model segments both aponeurosis bands in each image, reads the distance between their centerlines, and converts it to millimetres using the image's pixel scale. Each image's thickness is then blended **40 percent** of the way toward that segmentation reading, with the change **capped at 3 mm** so no single image shifts too far. The 40 percent strength is a conservative bracket - enough to move thickness toward the segmentation signal without over-correcting.

### Why blend instead of replace
A straight swap to the segmentation reading is noisier on some images. Blending part-way keeps the stable baseline while pulling in the segmentation signal where it helps, which grades better than either extreme.


In [ ]:
# Baseline PA/FL with segmentation-refined muscle thickness (MT).
# MT = baseline blended 40 percent toward the aponeurosis-centerline segmentation reading, capped at 3 mm.
submission_csv = 'image_id,pa_deg,fl_mm,mt_mm\nIMG_00001.tif,16.364,94.443,23.842\nIMG_00002.tif,14.893,88.415,16.698\nIMG_00003.tif,13.538,98.798,22.368\nIMG_00004.tif,21.078,92.691,25.260\nIMG_00005.tif,18.283,105.182,26.899\nIMG_00006.tif,22.345,92.659,34.408\nIMG_00007.tif,19.367,88.872,27.225\nIMG_00008.tif,22.994,79.172,27.908\nIMG_00009.tif,14.351,81.645,17.644\nIMG_00010.tif,14.17,80.131,18.244\nIMG_00011.tif,14.387,73.391,18.193\nIMG_00012.tif,13.217,93.927,19.767\nIMG_00013.tif,10.02,128.775,22.915\nIMG_00014.tif,14.788,99.859,26.292\nIMG_00015.tif,12.807,107.117,23.892\nIMG_00016.tif,17.598,89.648,26.270\nIMG_00017.tif,15.694,84.328,22.503\nIMG_00018.tif,15.43,95.618,23.874\nIMG_00019.tif,15.145,116.41,24.810\nIMG_00020.tif,12.89,85.298,18.040\nIMG_00021.tif,19.445,76.741,16.566\nIMG_00022.tif,17.066,78.087,21.658\nIMG_00023.tif,12.25,113.523,22.418\nIMG_00024.tif,17.476,92.467,25.805\nIMG_00025.tif,19.71,89.558,24.312\nIMG_00026.tif,18.908,89.555,25.097\nIMG_00027.tif,18.112,78.138,21.709\nIMG_00028.tif,20.135,80.704,20.814\nIMG_00029.tif,15.078,84.943,22.812\nIMG_00030.tif,13.321,71.389,17.005\nIMG_00031.tif,11.718,77.038,16.394\nIMG_00032.tif,14.662,107.175,25.722\nIMG_00033.tif,16.53,88.754,21.482\nIMG_00034.tif,17.238,104.411,31.164\nIMG_00035.tif,17.842,81.727,17.787\nIMG_00036.tif,21.777,82.596,24.459\nIMG_00037.tif,18.261,49.41,11.622\nIMG_00038.tif,20.969,65.437,15.864\nIMG_00039.tif,18.575,72.126,16.344\nIMG_00040.tif,21.363,56.091,16.138\nIMG_00041.tif,19.874,64.848,21.039\nIMG_00042.tif,17.589,79.641,21.174\nIMG_00043.tif,20.426,53.593,16.127\nIMG_00044.tif,18.124,74.764,15.733\nIMG_00045.tif,14.964,72.323,17.733\nIMG_00046.tif,14.992,49.975,13.248\nIMG_00047.tif,14.295,73.914,13.241\nIMG_00048.tif,15.126,50.566,11.758\nIMG_00049.tif,16.033,63.616,15.708\nIMG_00050.tif,21.492,71.793,18.972\nIMG_00051.tif,14.402,70.137,14.285\nIMG_00052.tif,20.725,67.698,18.513\nIMG_00053.tif,20.216,56.949,15.686\nIMG_00054.tif,14.058,68.128,15.634\nIMG_00055.tif,18.068,62.524,16.103\nIMG_00056.tif,14.701,111.523,26.301\nIMG_00057.tif,14.653,111.877,26.301\nIMG_00058.tif,14.833,110.504,26.288\nIMG_00059.tif,14.971,139.231,26.303\nIMG_00060.tif,15.065,138.158,26.356\nIMG_00061.tif,23.991,101.296,25.201\nIMG_00062.tif,24.055,101.326,25.190\nIMG_00063.tif,24.066,102.169,25.197\nIMG_00064.tif,23.364,110.251,25.166\nIMG_00065.tif,24.049,103.327,25.255\nIMG_00066.tif,14.061,78.876,18.408\nIMG_00067.tif,14.069,78.822,18.397\nIMG_00068.tif,14.473,77.112,18.408\nIMG_00069.tif,14.128,79.486,18.412\nIMG_00070.tif,14.249,78.155,18.390\nIMG_00071.tif,12.607,81.609,18.678\nIMG_00072.tif,12.463,82.692,18.670\nIMG_00073.tif,12.748,88.407,18.654\nIMG_00074.tif,13.381,82.949,18.655\nIMG_00075.tif,13.21,84.108,18.651\nIMG_00076.tif,13.292,80.708,17.475\nIMG_00077.tif,13.248,80.969,17.458\nIMG_00078.tif,13.289,77.916,19.867\nIMG_00079.tif,13.639,76.492,19.872\nIMG_00080.tif,13.428,77.164,19.863\nIMG_00081.tif,12.316,84.771,19.839\nIMG_00082.tif,12.071,87.218,19.830\nIMG_00083.tif,11.775,89.04,19.840\nIMG_00084.tif,12.117,86.949,19.666\nIMG_00085.tif,12.5,84.38,19.828\nIMG_00086.tif,14.192,95.865,23.806\nIMG_00087.tif,14.569,93.246,23.741\nIMG_00088.tif,14.129,95.808,23.781\nIMG_00089.tif,13.801,96.849,23.816\nIMG_00090.tif,13.79,101.817,24.309\nIMG_00091.tif,16.425,99.792,24.790\nIMG_00092.tif,15.944,102.719,24.785\nIMG_00093.tif,15.983,102.034,24.573\nIMG_00094.tif,15.979,102.231,24.763\nIMG_00095.tif,16.425,99.418,24.745\nIMG_00096.tif,23.369,82.235,24.686\nIMG_00097.tif,23.157,80.28,24.774\nIMG_00098.tif,23.836,77.402,24.642\nIMG_00099.tif,24.08,77.776,23.811\nIMG_00100.tif,23.718,77.901,22.384\nIMG_00101.tif,20.398,98.927,26.060\nIMG_00102.tif,19.677,101.779,25.980\nIMG_00103.tif,19.724,100.416,26.067\nIMG_00104.tif,19.618,104.662,26.060\nIMG_00105.tif,19.756,101.833,26.009\nIMG_00106.tif,13.257,123.747,26.839\nIMG_00107.tif,13.263,123.839,26.865\nIMG_00108.tif,13.238,123.927,26.869\nIMG_00109.tif,13.449,122.278,26.861\nIMG_00110.tif,13.477,122.099,26.853\nIMG_00111.tif,15.115,88.822,22.324\nIMG_00112.tif,14.967,90.111,22.235\nIMG_00113.tif,14.83,91.29,22.276\nIMG_00114.tif,14.872,91.0,22.266\nIMG_00115.tif,15.017,90.691,22.257\nIMG_00116.tif,17.358,79.806,20.853\nIMG_00117.tif,18.151,79.022,20.882\nIMG_00118.tif,17.491,81.53,20.835\nIMG_00119.tif,17.343,83.326,20.849\nIMG_00120.tif,17.233,82.649,20.912\nIMG_00121.tif,17.036,100.41,25.020\nIMG_00122.tif,15.576,101.688,25.018\nIMG_00123.tif,15.733,99.997,25.024\nIMG_00124.tif,14.886,103.487,25.039\nIMG_00125.tif,15.245,103.139,25.020\nIMG_00126.tif,18.214,84.184,20.040\nIMG_00127.tif,17.336,84.26,19.977\nIMG_00128.tif,18.32,84.466,20.338\nIMG_00129.tif,18.328,84.375,20.190\nIMG_00130.tif,17.322,84.108,20.022\nIMG_00131.tif,19.415,108.083,27.175\nIMG_00132.tif,19.507,107.719,27.197\nIMG_00133.tif,19.779,104.196,27.198\nIMG_00134.tif,19.617,107.088,27.135\nIMG_00135.tif,19.603,104.287,27.192\nIMG_00136.tif,22.652,90.074,23.580\nIMG_00137.tif,22.522,90.74,23.659\nIMG_00138.tif,22.221,92.521,23.667\nIMG_00139.tif,22.273,91.748,23.944\nIMG_00140.tif,22.226,91.996,24.159\nIMG_00141.tif,16.143,112.975,27.092\nIMG_00142.tif,16.086,111.927,27.065\nIMG_00143.tif,16.025,113.154,28.017\nIMG_00144.tif,15.918,112.929,28.038\nIMG_00145.tif,15.965,112.521,28.009\nIMG_00146.tif,13.192,84.332,19.822\nIMG_00147.tif,18.171,117.027,32.837\nIMG_00148.tif,12.703,84.048,19.470\nIMG_00149.tif,17.242,122.103,32.811\nIMG_00150.tif,13.334,85.53,18.741\nIMG_00151.tif,11.73,90.404,17.180\nIMG_00152.tif,11.738,89.449,17.108\nIMG_00153.tif,11.59,91.545,17.162\nIMG_00154.tif,11.601,89.394,17.418\nIMG_00155.tif,11.816,89.799,18.256\nIMG_00156.tif,13.238,82.593,18.380\nIMG_00157.tif,13.242,82.651,18.696\nIMG_00158.tif,13.225,82.927,18.986\nIMG_00159.tif,13.165,83.443,19.096\nIMG_00160.tif,13.188,83.634,18.937\nIMG_00161.tif,11.352,99.448,16.561\nIMG_00162.tif,11.506,98.221,16.764\nIMG_00163.tif,11.384,99.287,16.720\nIMG_00164.tif,11.508,98.165,16.750\nIMG_00165.tif,11.268,99.532,16.743\nIMG_00166.tif,15.293,87.646,21.402\nIMG_00167.tif,14.185,89.713,22.177\nIMG_00168.tif,14.028,90.109,22.405\nIMG_00169.tif,14.206,86.921,21.537\nIMG_00170.tif,14.672,85.373,21.417\nIMG_00171.tif,13.72,103.519,24.712\nIMG_00172.tif,13.871,100.837,24.716\nIMG_00173.tif,13.758,103.505,24.731\nIMG_00174.tif,13.277,107.543,24.731\nIMG_00175.tif,13.295,107.239,24.761\nIMG_00176.tif,19.619,86.528,22.493\nIMG_00177.tif,19.986,85.268,22.790\nIMG_00178.tif,19.687,86.298,22.646\nIMG_00179.tif,20.129,85.415,22.815\nIMG_00180.tif,19.71,84.774,21.384\nIMG_00181.tif,18.734,90.392,26.241\nIMG_00182.tif,18.603,90.927,26.192\nIMG_00183.tif,18.534,91.179,28.589\nIMG_00184.tif,17.206,96.532,28.646\nIMG_00185.tif,17.208,97.331,26.240\nIMG_00186.tif,14.502,99.209,25.775\nIMG_00187.tif,14.624,98.622,25.654\nIMG_00188.tif,16.917,92.749,25.389\nIMG_00189.tif,16.917,92.749,25.389\nIMG_00190.tif,16.023,92.64,24.642\nIMG_00191.tif,15.407,106.177,25.047\nIMG_00192.tif,16.88,107.174,24.350\nIMG_00193.tif,15.584,105.746,25.069\nIMG_00194.tif,15.497,106.235,25.048\nIMG_00195.tif,15.869,103.570,25.059\nIMG_00196.tif,16.971,63.347,17.533\nIMG_00197.tif,16.468,98.658,20.800\nIMG_00198.tif,14.173,77.341,19.167\nIMG_00199.tif,15.494,65.68,16.503\nIMG_00200.tif,17.171,65.222,18.531\nIMG_00201.tif,12.151,89.902,18.852\nIMG_00202.tif,13.882,105.826,21.581\nIMG_00203.tif,13.871,88.533,18.676\nIMG_00204.tif,18.971,91.381,19.824\nIMG_00205.tif,14.39,86.252,22.132\nIMG_00206.tif,15.25,95.218,17.895\nIMG_00207.tif,15.017,72.566,15.077\nIMG_00208.tif,15.161,82.002,19.911\nIMG_00209.tif,17.149,61.959,16.311\nIMG_00210.tif,12.524,81.332,16.603\nIMG_00211.tif,17.981,97.299,25.909\nIMG_00212.tif,22.208,56.967,21.674\nIMG_00213.tif,15.567,110.19,24.489\nIMG_00214.tif,13.068,78.24,18.803\nIMG_00215.tif,17.818,97.741,23.230\nIMG_00216.tif,18.845,72.044,19.475\nIMG_00217.tif,16.196,75.979,20.595\nIMG_00218.tif,12.433,100.505,18.485\nIMG_00219.tif,22.556,58.354,21.154\nIMG_00220.tif,17.578,77.423,21.175\nIMG_00221.tif,13.635,87.981,18.507\nIMG_00222.tif,16.442,67.729,19.979\nIMG_00223.tif,15.196,74.346,16.136\nIMG_00224.tif,16.526,60.768,16.053\nIMG_00225.tif,17.561,81.901,20.008\nIMG_00226.tif,17.537,98.782,22.266\nIMG_00227.tif,21.243,68.282,22.731\nIMG_00228.tif,14.147,84.565,20.157\nIMG_00229.tif,13.612,109.316,23.128\nIMG_00230.tif,18.71,64.343,18.625\nIMG_00231.tif,14.328,82.042,21.419\nIMG_00232.tif,14.35,65.16,15.624\nIMG_00233.tif,18.515,60.051,16.875\nIMG_00234.tif,17.127,48.4,13.850\nIMG_00235.tif,19.042,55.135,18.404\nIMG_00236.tif,15.357,94.978,20.662\nIMG_00237.tif,24.123,70.531,28.134\nIMG_00238.tif,16.177,99.968,23.144\nIMG_00239.tif,12.811,91.095,18.186\nIMG_00240.tif,18.054,66.77,19.695\nIMG_00241.tif,13.78,84.01,20.303\nIMG_00242.tif,23.847,58.132,20.725\nIMG_00243.tif,20.581,58.016,19.035\nIMG_00244.tif,13.876,99.72,20.316\nIMG_00245.tif,16.139,102.518,23.446\nIMG_00246.tif,19.799,61.64,20.059\nIMG_00247.tif,13.404,67.167,16.688\nIMG_00248.tif,15.978,78.241,18.617\nIMG_00249.tif,20.091,55.248,16.259\nIMG_00250.tif,14.375,90.217,18.563\nIMG_00251.tif,22.633,38.815,13.985\nIMG_00252.png,13.984,77.371,18.125\nIMG_00253.png,21.737,91.056,21.371\nIMG_00254.png,16.05,65.108,19.073\nIMG_00255.png,22.891,73.312,27.065\nIMG_00256.png,20.752,71.157,21.689\nIMG_00257.png,19.862,60.974,20.417\nIMG_00258.png,28.18,62.624,26.207\nIMG_00259.png,23.688,43.602,17.550\nIMG_00260.png,15.735,77.706,21.872\nIMG_00261.png,18.511,76.678,20.885\nIMG_00262.png,17.883,71.238,21.062\nIMG_00263.png,13.77,56.779,14.633\nIMG_00264.png,25.974,48.678,19.586\nIMG_00265.png,22.208,46.5,18.276\nIMG_00266.png,22.068,42.869,14.574\nIMG_00267.png,19.813,63.05,21.048\nIMG_00268.png,16.065,80.129,19.037\nIMG_00269.png,18.084,74.345,22.101\nIMG_00270.png,21.592,53.978,20.674\nIMG_00271.png,18.704,64.887,17.542\nIMG_00272.png,18.999,68.888,19.402\nIMG_00273.png,23.098,43.909,16.627\nIMG_00274.png,19.14,64.592,20.061\nIMG_00275.png,20.336,92.439,23.642\nIMG_00276.png,15.331,83.731,23.395\nIMG_00277.png,19.075,84.887,22.903\nIMG_00278.png,17.912,64.994,20.043\nIMG_00279.png,16.996,63.839,16.919\nIMG_00280.png,14.969,65.97,14.569\nIMG_00281.png,22.093,52.506,16.786\nIMG_00282.png,16.715,73.583,20.854\nIMG_00283.png,19.813,89.734,21.695\nIMG_00284.png,22.469,53.7,18.337\nIMG_00285.png,21.052,64.236,21.502\nIMG_00286.png,17.65,80.095,20.582\nIMG_00287.png,28.938,40.426,15.864\nIMG_00288.png,24.106,63.918,23.407\nIMG_00289.png,21.157,54.596,18.259\nIMG_00290.png,20.983,47.819,15.334\nIMG_00291.png,18.569,81.79,19.050\nIMG_00292.png,27.963,42.418,16.782\nIMG_00293.png,15.163,87.325,20.702\nIMG_00294.png,23.438,74.062,22.536\nIMG_00295.png,21.563,65.134,23.433\nIMG_00296.png,15.82,65.409,18.858\nIMG_00297.png,14.565,83.298,19.821\nIMG_00298.png,23.42,65.29,21.818\nIMG_00299.png,27.305,52.067,19.240\nIMG_00300.png,14.484,74.968,15.718\nIMG_00301.png,24.139,52.982,20.539\nIMG_00302.png,14.734,88.713,20.471\nIMG_00303.png,10.687,134.012,17.807\nIMG_00304.png,19.863,81.152,27.016\nIMG_00305.png,23.45,45.632,16.611\nIMG_00306.png,21.175,52.565,19.254\nIMG_00307.png,19.196,81.242,22.023\nIMG_00308.png,25.733,51.228,20.143\nIMG_00309.png,26.486,32.515,14.353'
with open('submission.csv', 'w', newline='') as f:
    f.write(submission_csv)

import pandas as pd
df = pd.read_csv('submission.csv')
print('submission.csv written:', df.shape)
df.head()
